In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('Join').getOrCreate()

25/03/16 02:50:58 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
spark.sparkContext.applicationId

'application_1742093195679_0005'

In [4]:
spark.conf.set('spark.sql.adaptive.enabled','false')

In [5]:
bucket = 'gs://dataproc-staging-us-central1-458263062208-tw36mmqt'
file_path = '/notebooks/jupyter/Data/'

In [6]:
customers_df = spark.read.csv(bucket+file_path+'customers.csv',header=True,inferSchema=True)
orders_df = spark.read.csv(bucket+file_path+'orders.csv',header=True,inferSchema=True)

In [8]:
customers_df.show(2),orders_df.show(2)

+-----------+----------+-------+-----------+-------+-------------------+---------+
|customer_id|      name|   city|      state|country|  registration_date|is_active|
+-----------+----------+-------+-----------+-------+-------------------+---------+
|          0|Customer_0| Mumbai|  Telangana|  India|2023-03-21 00:00:00|     true|
|          1|Customer_1|Chennai|West Bengal|  India|2023-05-27 00:00:00|    false|
+-----------+----------+-------+-----------+-------+-------------------+---------+
only showing top 2 rows

+--------+-----------+-------------------+-----------------+-------+
|order_id|customer_id|         order_date|     total_amount| status|
+--------+-----------+-------------------+-----------------+-------+
|       0|    3194509|2024-04-12 00:00:00|772.3507972244216|Shipped|
|       1|    8003925|2024-05-06 00:00:00| 404.843827081314|Shipped|
+--------+-----------+-------------------+-----------------+-------+
only showing top 2 rows



(None, None)

In [9]:
!hadoop fs -ls -h /data2/

Found 1 items
-rw-r--r--   2 tech_mayankagg hadoop      1.0 M 2025-03-01 03:34 /data2/customers.csv


In [10]:
customers_small_df = spark.read.csv('/data2/customers.csv',header=True,inferSchema=True)

In [11]:
customers_small_df.show(2)

+-----------+----------+---------+-----------+-------+-------------------+---------+
|customer_id|      name|     city|      state|country|  registration_date|is_active|
+-----------+----------+---------+-----------+-------+-------------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|2023-06-29 00:00:00|    false|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|2023-12-07 00:00:00|     true|
+-----------+----------+---------+-----------+-------+-------------------+---------+
only showing top 2 rows



In [12]:
joined_df = orders_df.join(customers_small_df,on='customer_id',how='inner')

We seem to be working with identical names which is always not the case.

In [13]:
# Don't do show() as that might take less partition

joined_df.write.format('noop').mode('overwrite').save()

In [14]:
customers_small_df.count()

17653

In [ ]:
spark.conf.set('spark.sql.adaptive.enabled','false')


In [16]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold',-1)

In [17]:
spark.conf.get('spark.sql.autoBroadcastJoinThreshold')

'-1'

In [18]:
orders_df.join(customers_small_df,on='customer_id',how='inner').write.format('noop').mode('overwrite').save()

Hi Mayank , I didn't receive any mail today so thought there was no class today! later on came to know class was there. Could you let me know what are we covering today?

In [24]:
orders_df.selectExpr('customer_id', 'customer_id % 200 as cust_key').show()
customers_small_df.selectExpr('customer_id', 'customer_id % 200 as cust_key').show()

+-----------+--------+
|customer_id|cust_key|
+-----------+--------+
|    3194509|     109|
|    8003925|     125|
|    1946602|       2|
|    8511049|      49|
|    5013836|      36|
|    5352874|      74|
|    3676047|      47|
|     663456|      56|
|    5471172|     172|
|    8298598|     198|
|    2217320|     120|
|    4796277|      77|
|     766907|     107|
|    3924592|     192|
|    7014407|       7|
|    4008694|      94|
|    8248295|      95|
|     372605|       5|
|    7289017|      17|
|    6406178|     178|
+-----------+--------+
only showing top 20 rows

+-----------+--------+
|customer_id|cust_key|
+-----------+--------+
|          0|       0|
|          1|       1|
|          2|       2|
|          3|       3|
|          4|       4|
|          5|       5|
|          6|       6|
|          7|       7|
|          8|       8|
|          9|       9|
|         10|      10|
|         11|      11|
|         12|      12|
|         13|      13|
|         14|      14|
|       

In [26]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold','50m')

In [27]:
spark.conf.get('spark.sql.autoBroadcastJoinThreshold')

'50m'

In [28]:
orders_df.join(customers_small_df,on='customer_id',how='inner').write.format('noop').mode('overwrite').save()

# Skew Join

In [7]:
orders_df.count(),customers_df.count()

(8767358, 8767358)

In [8]:
customers_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: timestamp (nullable = true)
 |-- is_active: boolean (nullable = true)



In [9]:
import pyspark.sql.functions as F

In [10]:
customers_df.groupBy('city').count().show()

+---------+-------+
|     city|  count|
+---------+-------+
|Bangalore|1094195|
|  Chennai|1095052|
|   Mumbai|1095815|
|Ahmedabad|1097162|
|  Kolkata|1096777|
|     Pune|1095748|
|    Delhi|1096183|
|Hyderabad|1096426|
+---------+-------+



In [11]:
skew_factor = 100

In [12]:
mumbai_df = customers_df.filter(F.col('city')=='Mumbai')

In [41]:
mumbai_df.count()

1095815

In [13]:
# Generate multiple copies of Mumbai's data using an array expansion trick
replicated_mumbai_df = mumbai_df.withColumn("repeat", F.explode(F.array([F.lit(i) for i in range(skew_factor)])))
replicated_mumbai_df.count()

109581500

In [14]:


# Adjust customer_id to avoid duplicates
replicated_mumbai_df = replicated_mumbai_df.withColumn("customer_id",(F.col("customer_id") + F.monotonically_increasing_id()) % (10**8)).drop("repeat")
# Combine the skewed Mumbai data back into the main dataset
customers_big_customers_dfskewed_df = customers_df.union(replicated_mumbai_df)
# Verify the new distribution
customers_big_customers_dfskewed_df.groupBy("city").count().show()

+---------+---------+
|     city|    count|
+---------+---------+
|Bangalore|  1094195|
|  Chennai|  1095052|
|   Mumbai|110677315|
|Ahmedabad|  1097162|
|  Kolkata|  1096777|
|     Pune|  1095748|
|    Delhi|  1096183|
|Hyderabad|  1096426|
+---------+---------+



In [15]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold',-1)
spark.conf.set('spark.sql.adaptive.enabled','false')

In [16]:
city_data = [
    ("Mumbai", "High Population"),
    ("Bangalore", "IT Hub"),
    ("Hyderabad", "Tech City"),
    ("Delhi", "Capital"),
    ("Chennai", "Coastal City"),
    ("Pune", "Education Hub"),
    ("Ahmedabad", "Business Center"),
    ("Kolkata", "Cultural City"),
]

city_columns = ["city", "description"]
city_df = spark.createDataFrame(city_data, city_columns)


In [17]:
city_df.count()

8

In [18]:
joined_df = customers_big_customers_dfskewed_df.join(city_df,'city','inner')

In [19]:
joined_df.write.format('noop').mode('overwrite').save()

explain the term skewness with example

Our executor has storage memory as well, right? Why can’t the job get successful by spilling data to disk?

In [28]:
from pyspark.sql.functions import floor,lit,col,rand,concat

In [46]:
salt_range  = 30

In [47]:
customers_salted = customers_big_customers_dfskewed_df.withColumn(
    'salt',(F.floor(rand()*salt_range)).cast('int')
).withColumn('city_salted',concat(col('city'),F.lit('_'),col('salt')))

In [48]:
customers_salted.show()

+-----------+-----------+---------+-----------+-------+-------------------+---------+----+------------+
|customer_id|       name|     city|      state|country|  registration_date|is_active|salt| city_salted|
+-----------+-----------+---------+-----------+-------+-------------------+---------+----+------------+
|          0| Customer_0|   Mumbai|  Telangana|  India|2023-03-21 00:00:00|     true|  10|   Mumbai_10|
|          1| Customer_1|  Chennai|West Bengal|  India|2023-05-27 00:00:00|    false|   9|   Chennai_9|
|          2| Customer_2|     Pune|  Karnataka|  India|2023-10-11 00:00:00|    false|  26|     Pune_26|
|          3| Customer_3|Hyderabad|    Gujarat|  India|2023-11-11 00:00:00|    false|  27|Hyderabad_27|
|          4| Customer_4|   Mumbai|  Karnataka|  India|2023-05-09 00:00:00|    false|  19|   Mumbai_19|
|          5| Customer_5|Bangalore|  Karnataka|  India|2023-03-12 00:00:00|    false|   1| Bangalore_1|
|          6| Customer_6|Hyderabad| Tamil Nadu|  India|2023-07-1

In [32]:
city_df.show()

+---------+---------------+
|     city|    description|
+---------+---------------+
|   Mumbai|High Population|
|Bangalore|         IT Hub|
|Hyderabad|      Tech City|
|    Delhi|        Capital|
|  Chennai|   Coastal City|
|     Pune|  Education Hub|
|Ahmedabad|Business Center|
|  Kolkata|  Cultural City|
+---------+---------------+



In [49]:
from pyspark.sql.functions import explode,array

from pyspark.sql.functions import explode, array, lit

city_df_expanded = city_df.withColumn(
    'salt',
    explode(array(*[lit(i) for i in range(salt_range)]))
).withColumn('city_salted',concat(col('city'),F.lit('_'),col('salt')))



In [51]:
city_df_expanded.show(20)

+------+---------------+----+-----------+
|  city|    description|salt|city_salted|
+------+---------------+----+-----------+
|Mumbai|High Population|   0|   Mumbai_0|
|Mumbai|High Population|   1|   Mumbai_1|
|Mumbai|High Population|   2|   Mumbai_2|
|Mumbai|High Population|   3|   Mumbai_3|
|Mumbai|High Population|   4|   Mumbai_4|
|Mumbai|High Population|   5|   Mumbai_5|
|Mumbai|High Population|   6|   Mumbai_6|
|Mumbai|High Population|   7|   Mumbai_7|
|Mumbai|High Population|   8|   Mumbai_8|
|Mumbai|High Population|   9|   Mumbai_9|
|Mumbai|High Population|  10|  Mumbai_10|
|Mumbai|High Population|  11|  Mumbai_11|
|Mumbai|High Population|  12|  Mumbai_12|
|Mumbai|High Population|  13|  Mumbai_13|
|Mumbai|High Population|  14|  Mumbai_14|
|Mumbai|High Population|  15|  Mumbai_15|
|Mumbai|High Population|  16|  Mumbai_16|
|Mumbai|High Population|  17|  Mumbai_17|
|Mumbai|High Population|  18|  Mumbai_18|
|Mumbai|High Population|  19|  Mumbai_19|
+------+---------------+----+-----

In [42]:
customers_salted.join(city_df_expanded,'city','inner').write.format('noop').mode('overwrite').save()

25/03/16 03:35:49 ERROR OverwriteByExpressionExec: Data source write support org.apache.spark.sql.execution.datasources.noop.NoopBatchWrite$@f09fc0 is aborting.
25/03/16 03:35:49 ERROR OverwriteByExpressionExec: Data source write support org.apache.spark.sql.execution.datasources.noop.NoopBatchWrite$@f09fc0 aborted.
25/03/16 03:35:49 WARN TaskSetManager: Lost task 51.0 in stage 49.0 (TID 714) (my-cluster-w-0.us-central1-c.c.celtic-science-452211-u6.internal executor 4): TaskKilled (Stage cancelled)


Py4JJavaError: An error occurred while calling o312.save.
: org.apache.spark.SparkException: Writing job aborted
	at org.apache.spark.sql.errors.QueryExecutionErrors$.writingJobAbortedError(QueryExecutionErrors.scala:766)
	at org.apache.spark.sql.execution.datasources.v2.V2TableWriteExec.writeWithV2(WriteToDataSourceV2Exec.scala:409)
	at org.apache.spark.sql.execution.datasources.v2.V2TableWriteExec.writeWithV2$(WriteToDataSourceV2Exec.scala:353)
	at org.apache.spark.sql.execution.datasources.v2.OverwriteByExpressionExec.writeWithV2(WriteToDataSourceV2Exec.scala:262)
	at org.apache.spark.sql.execution.datasources.v2.V2ExistingTableWriteExec.run(WriteToDataSourceV2Exec.scala:332)
	at org.apache.spark.sql.execution.datasources.v2.V2ExistingTableWriteExec.run$(WriteToDataSourceV2Exec.scala:331)
	at org.apache.spark.sql.execution.datasources.v2.OverwriteByExpressionExec.run(WriteToDataSourceV2Exec.scala:262)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.result$lzycompute(

In [52]:
customers_salted.join(city_df_expanded,'city_salted','inner').write.format('noop').mode('overwrite').save()

Mayank, can you give us other scenarios where data skewness affects the performance apart from during join operation. and how should we tackle that, should we use salting or bucketing itself?

Will AQE help in anyway as it wouldn’t have default 200 partitions ?

In [53]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold',-1)
spark.conf.set('spark.sql.adaptive.enabled','true')

In [54]:
customers_salted.join(city_df_expanded,'city','inner').write.format('noop').mode('overwrite').save()

why do we need bucketing , if spark is doing its own kind of bucketing using a hash key , in bucketing as well we are creating buckets on basis of a column

so bucketing is done while writing onto disk or memory and hash-based partitioning is working in runtime to decide which keys goes to which partition

In salting, can we have an issue of cross join for some salted keys?

Do we need to specify any parameters in spark to read the data if its partititioned ?

# Project

This dataset contains information about candidates, committees, PACs (political action committees), House & Senate campaigns & about transactions, contributions by individuals & committees & operating & independent expenditures for US elections from Jan 1, 2019 to May 2020.




This dataset was created by downloading the bulk dataset from the FEC website page here & then, creating csv files using the correct headers. You will also find data description links from the bulk dataset FEC page linked above.



2023–2024,2021–2022,2019–2020

https://www.fec.gov/data/browse-data/?tab=bulk-data

4th April

Will be sending a mail to make things more clear, don't worry and don't hate me.

just trying to emulate real company data and project as I promised, sorry cannot make it easier

In [58]:
customers_salted.explain(True)

== Parsed Logical Plan ==
'Project [customer_id#151L, name#18, city#19, state#20, country#21, registration_date#22, is_active#23, salt#538, concat('city, _, 'salt) AS city_salted#547]
+- Project [customer_id#151L, name#18, city#19, state#20, country#21, registration_date#22, is_active#23, cast(FLOOR((rand(1878025399374599499) * cast(30 as double))) as int) AS salt#538]
   +- Union false, false
      :- Project [cast(customer_id#17 as bigint) AS customer_id#151L, name#18, city#19, state#20, country#21, registration_date#22, is_active#23]
      :  +- Relation [customer_id#17,name#18,city#19,state#20,country#21,registration_date#22,is_active#23] csv
      +- Project [customer_id#128L, name#145, city#146, state#147, country#148, registration_date#149, is_active#150]
         +- Project [((cast(customer_id#144 as bigint) + monotonically_increasing_id()) % cast(100000000 as bigint)) AS customer_id#128L, name#145, city#146, state#147, country#148, registration_date#149, is_active#150, repeat#

In [59]:
customers_salted.createOrReplaceGlobalTempView('customer_salted')

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


In [61]:
customers_salted.createOrReplaceTempView('customers_salted_temp')

In [69]:
spark.sql('select city from customers_salted_temp').explain(True)

== Parsed Logical Plan ==
'Project ['city]
+- 'UnresolvedRelation [customers_salted_temp], [], false

== Analyzed Logical Plan ==
city: string
Project [city#19]
+- SubqueryAlias customers_salted_temp
   +- View (`customers_salted_temp`, [customer_id#151L,name#18,city#19,state#20,country#21,registration_date#22,is_active#23,salt#538,city_salted#547])
      +- Project [customer_id#151L, name#18, city#19, state#20, country#21, registration_date#22, is_active#23, salt#538, concat(city#19, _, cast(salt#538 as string)) AS city_salted#547]
         +- Project [customer_id#151L, name#18, city#19, state#20, country#21, registration_date#22, is_active#23, cast(FLOOR((rand(1878025399374599499) * cast(30 as double))) as int) AS salt#538]
            +- Union false, false
               :- Project [cast(customer_id#17 as bigint) AS customer_id#151L, name#18, city#19, state#20, country#21, registration_date#22, is_active#23]
               :  +- Relation [customer_id#17,name#18,city#19,state#20,coun

In [ ]:
spark.sql('select city from customers_salted_temp').explain(True)

In [71]:
city_df_expanded.write.parquet('/data/city_df_expanded')